In [8]:
import pandas as pd
import numpy as np
import uuid 
from itertools import combinations

# --- 1. MOCK DATA SETUP ---
# To test Stage 2, we need enriched Bank data and the Book data.
# These mock datasets intentionally contain common break scenarios.

# 1.1 FX Fee Tolerance Rules (From fx_fees.json)
FX_FEE_TOLERANCES = {
    'USD': 30.00,
    'EUR': 24.00,
    'GBP': 20.00,
    'CAD': 40.00
}

# 1.2 Mock Enriched Bank Data (Output of Stage 1)
def create_mock_enriched_bank_data():
    """Simulates enriched bank data ready for matching, including an N:M case."""
    data = {
        'Transaction ID': ['BTXN01', 'BTXN02', 'BTXN03', 'BTXN04', 'BTXN05', 'BTXN06', 'BTXN07', 'BTXN08', 'BTXN09'],
        'Fund ID': ['50360414', '50342304', '50360414', '50342304', '50360414', '50342304', '50360414', '50360414', '50342304'],
        'Currency': ['USD', 'EUR', 'USD', 'GBP', 'USD', 'EUR', 'USD', 'USD', 'EUR'],
        'Amount': [-9970.00, 499976.00, 1000.00, -50000.00, -8000.00, 25000.00, 950.00, 0.00, -15000.00], # BTXN09 is the 1-side of 1:2
        'Tran Type': ['TRDL', 'RCPT', 'RCPT', 'TRDL', 'TRDL', 'RCPT', 'RCPT', 'JRNL', 'TRDL'],
        'Deal ID': ['D1000007', 'D1000029', 'D1000015', 'D1000004', 'D1000040', 'D1000007', 'D1000015', None, 'D1000029'],
        'Product ID': ['PID00006', 'PID00028', 'PID00014', 'PID00003', 'PID00039', 'PID00006', 'PID00014', None, 'PID00028'],
        'Payment Date': pd.to_datetime(['2025-06-15', '2025-06-15', '2025-06-18', '2025-06-20', '2025-06-25', '2025-06-26', '2025-06-28', '2025-06-29', '2025-07-05']),
        'Enrichment Request status': ['Auto-Enriched - Rule Match'] * 8 + ['Auto-Enriched - Rule Match']
    }
    df = pd.DataFrame(data)
    df['Bank Amount'] = df['Amount'].abs()
    df['Tran Type_Bank'] = df['Tran Type']
    
    return df.drop(columns=['Amount', 'Tran Type']) 

# 1.3 Mock Book Data (WSO)
def create_mock_book_data():
    """Simulates WSO Book data with intentional breaks, including N:M splits."""
    data = {
        'Book ID': ['BXN01', 'BXN02', 'BXN03', 'BXN04', 'BXN05', 'BXN06', 'BXN07', 'BXN08', 'BXN09', 'BXN10', 'BXN11'],
        'Fund ID': ['50360414', '50342304', '50360414', '50342304', '50360414', '50342304', '50360414', '50360414', '50360414', '50342304', '50342304'],
        'Currency': ['USD', 'EUR', 'USD', 'GBP', 'USD', 'EUR', 'USD', 'USD', 'USD', 'EUR', 'EUR'],
        # BXN01 (Fee Break), BXN02 (Time Lag Break), BXN10/BXN11 (N:M Split)
        'Book Amount': [10000.00, 500000.00, 1000.00, 50000.00, 8000.00, 25000.00, 1000.00, 500.00, 500.00, 7500.00, 7500.00],
        'Product IDs': ['PID00006', 'PID00028', 'PID00014', 'PID00003', 'PID00039', 'PID00006', 'PID00014', 'PID00006', 'PID00006', 'PID00028', 'PID00028'],
        'Deal ID': ['D1000007', 'D1000029', 'D1000015', 'D1000004', 'D1000040', 'D1000007', 'D1000014', 'D1000015', 'D1000015', 'D1000029', 'D1000029'],
        'Payment Date': pd.to_datetime(['2025-06-15', '2025-06-16', '2025-06-18', '2025-06-23', '2025-06-24', '2025-06-26', '2025-07-01', '2025-06-15', '2025-06-15', '2025-07-05', '2025-07-05']),
    }
    return pd.DataFrame(data)

# --- N:M MATCHING FUNCTION ---

def find_n_to_m_matches(unmatched_bank, unmatched_book, bank_matched_ids, book_matched_ids):
    """
    Attempts to find 1-to-2 or 2-to-1 matches based on summing amounts.
    This is a simplification of the complex Subset Sum Problem.
    """
    print("   - Searching for 1:N Aggregation Matches (1 Bank -> N Book)...")
    n_to_m_matches = []
    
    # Exclude already matched transactions
    bank_to_check = unmatched_bank[~unmatched_bank['Transaction ID'].isin(bank_matched_ids)].copy()
    book_to_check = unmatched_book[~unmatched_book['Book ID'].isin(book_matched_ids)].copy()

    # Define common keys for aggregation matching
    group_keys = ['Fund ID', 'Deal ID', 'Currency']

    # 1. 1 Bank to N Book Match (e.g., Bank transaction is a total payment of multiple Book entries)
    for _, bank_row in bank_to_check.iterrows():
        bank_id = bank_row['Transaction ID']
        target_amount = bank_row['Bank Amount']
        
        # Filter potential book matches by key identifiers and date window
        potential_book = book_to_check[
            (book_to_check['Fund ID'] == bank_row['Fund ID']) &
            (book_to_check['Deal ID'] == bank_row['Deal ID']) &
            (book_to_check['Currency'] == bank_row['Currency']) &
            (abs(book_to_check['Payment Date'] - bank_row['Payment Date']).dt.days <= 3) # Use time lag window
        ].copy()
        
        if len(potential_book) < 2:
            continue
            
        # Check combinations of 2 or 3 book entries that sum to the bank amount
        for n in [2, 3]:
            # Convert to tuples for the combination logic
            book_tuples = list(potential_book.itertuples())
            
            for combo in combinations(book_tuples, n):
                # The named tuple structure is: Index (0), Book ID (1), Fund ID (2), Currency (3), Book Amount (4), ...
                # FIX: Use positional index 4 for 'Book Amount' (as done previously)
                combo_sum = sum(item[4] for item in combo)
                
                if abs(combo_sum - target_amount) < 0.01:
                    # Found a match!
                    # FIX: Use positional index 1 for 'Book ID'
                    book_ids = [item[1] for item in combo]
                    book_amounts = [item[4] for item in combo] 
                    
                    n_to_m_matches.append({
                        'Transaction ID': bank_id,
                        'Book IDs': book_ids,
                        'Bank Amount': target_amount,
                        'Book Amount': combo_sum,
                        'Status': f'Auto Matched - 1:{n} Aggregation'
                    })
                    
                    # Add IDs to the exclusion sets
                    bank_matched_ids.add(bank_id)
                    book_matched_ids.update(book_ids)
                    
                    # Stop searching for this bank transaction
                    break 
            if bank_id in bank_matched_ids:
                break
                
    return n_to_m_matches


# --- 2. STAGE 2: CASH CONTROL RECONCILIATION ---

def run_stage_2_matching(bank_df, book_df, fee_tolerances):
    """
    Executes a hierarchical matching process between enriched bank data and book data.
    """
    
    # 2.1 INITIAL SETUP: PREPARE AND MERGE DATA
    # --------------------------------------------------------------------------
    
    # Filter out transactions that were already flagged as non-deal/journal entries in Stage 1
    bank_matchable = bank_df[bank_df['Enrichment Request status'] != 'Filtered - No Enrichment Required'].copy()
    
    # Create a full join to find potential matches and breaks
    # We match on Fund ID and Deal ID initially
    merged_df = pd.merge(
        bank_matchable,
        book_df,
        on=['Fund ID', 'Deal ID', 'Currency'],
        how='left',
        suffixes=('_Bank', '_Book')
    )
    
    # Initialize Status and Break ID
    merged_df['Status'] = 'Unmatched'
    merged_df['Break ID'] = merged_df.apply(lambda row: str(uuid.uuid4()) if row['Book ID'] is None else None, axis=1)
    
    # Calculate Difference (Book - Bank)
    merged_df['Difference'] = merged_df['Book Amount'] - merged_df['Bank Amount']
    merged_df['Absolute Value Difference'] = merged_df['Difference'].abs()
    
    # Track which bank and book records have been matched
    bank_matched_ids = set()
    book_matched_ids = set()
    
    # --------------------------------------------------------------------------
    # STEP 2.2: EXACT MATCHING (Highest Confidence)
    # --------------------------------------------------------------------------
    
    print("-> Running 1. Exact Match Pass...")
    
    is_exact_match = (
        (merged_df['Difference'].abs() < 0.01) & # Zero difference
        (merged_df['Payment Date_Bank'] == merged_df['Payment Date_Book']) # Exact date
    )
    
    exact_matches = merged_df[is_exact_match].copy()
    
    if not exact_matches.empty:
        merged_df.loc[exact_matches.index, 'Status'] = 'Auto Matched - Exact'
        bank_matched_ids.update(exact_matches['Transaction ID'].unique())
        book_matched_ids.update(exact_matches['Book ID'].unique())
        print(f"   - Found {len(exact_matches)} exact matches.")
        
    # --------------------------------------------------------------------------
    # STEP 2.3: FUZZY/TOLERANCE MATCHING (Bank Fees)
    # --------------------------------------------------------------------------
    
    print("-> Running 2. Fuzzy/Tolerance Match Pass (Bank Fees)...")
    
    unmatched_df = merged_df[~merged_df['Transaction ID'].isin(bank_matched_ids)].copy()
    
    def check_fee_tolerance(row):
        """Checks if the difference equals the currency-specific bank fee for TRDL."""
        if row.get('Status') != 'Unmatched': return False
        currency = row.get('Currency')
        required_difference = fee_tolerances.get(currency)
        if required_difference is None: return False
            
        # Check if difference equals the fee AND it was a TRDL (Funding/Outflow)
        if (abs(row.get('Difference') - required_difference) < 0.01) and (row.get('Tran Type_Bank') == 'TRDL'):
            return True
        return False

    is_fuzzy_fee_match = unmatched_df.apply(check_fee_tolerance, axis=1)
    fuzzy_fee_matches = unmatched_df[is_fuzzy_fee_match].copy()

    if not fuzzy_fee_matches.empty:
        merged_df.loc[fuzzy_fee_matches.index, 'Status'] = 'Auto Matched - Fee Tolerance'
        bank_matched_ids.update(fuzzy_fee_matches['Transaction ID'].unique())
        book_matched_ids.update(fuzzy_fee_matches['Book ID'].unique())
        print(f"   - Found {len(fuzzy_fee_matches)} fee tolerance matches.")

    # --------------------------------------------------------------------------
    # STEP 2.4: TIME-LAG MATCHING (Within N days)
    # --------------------------------------------------------------------------
    
    print("-> Running 3. Time-Lag Match Pass (3-day window)...")
    
    unmatched_df = merged_df[~merged_df['Transaction ID'].isin(bank_matched_ids)].copy()

    is_time_lag_match = (
        (unmatched_df['Difference'].abs() < 0.01) & 
        (abs(unmatched_df['Payment Date_Bank'] - unmatched_df['Payment Date_Book']).dt.days <= 3) & 
        (unmatched_df['Status'] == 'Unmatched')
    )
    
    time_lag_matches = unmatched_df[is_time_lag_match].copy()
    
    if not time_lag_matches.empty:
        time_lag_matches = time_lag_matches.drop_duplicates(subset=['Transaction ID'], keep='first')
        
        merged_df.loc[time_lag_matches.index, 'Status'] = 'Auto Matched - Time Lag'
        bank_matched_ids.update(time_lag_matches['Transaction ID'].unique())
        book_matched_ids.update(time_lag_matches['Book ID'].unique())
        print(f"   - Found {len(time_lag_matches)} time-lag matches.")
        
    # --------------------------------------------------------------------------
    # STEP 2.5: N:M (AGGREGATION) MATCHING
    # --------------------------------------------------------------------------
    
    print("-> Running 4. N:M Aggregation Match Pass (1:2 or 1:3)...")

    # Get the transactions still unmatched after the strict passes
    final_unmatched_bank = bank_matchable[~bank_matchable['Transaction ID'].isin(bank_matched_ids)].copy()
    final_unmatched_book = book_df[~book_df['Book ID'].isin(book_matched_ids)].copy()
    
    # We must operate on the original (not merged) DataFrames for N:M to avoid merging complications
    n_to_m_results = find_n_to_m_matches(final_unmatched_bank, final_unmatched_book, bank_matched_ids, book_matched_ids)
    
    if n_to_m_results:
        # Update the status in the main merged_df for the matched Bank records
        for match in n_to_m_results:
            bank_id = match['Transaction ID']
            merged_df.loc[merged_df['Transaction ID'] == bank_id, 'Status'] = match['Status']
            # Note: For N:M matches, we keep the original break records in the merged_df, 
            # but mark the Bank transaction as matched. The Book side is handled during reporting.
        print(f"   - Found {len(n_to_m_results)} aggregation matches.")


    # --------------------------------------------------------------------------
    # STEP 2.6: FINAL BREAKS AND OUTPUT GENERATION
    # --------------------------------------------------------------------------
    
    # Identify transactions that are still 'Unmatched' after all passes
    final_breaks_df = merged_df[
        (merged_df['Status'] == 'Unmatched') & 
        (~merged_df['Transaction ID'].isin(bank_matched_ids))
    ].copy()
    
    # Clean up breaks (only keep one entry per unique Bank Transaction ID)
    final_breaks_df = final_breaks_df.drop_duplicates(subset=['Transaction ID'], keep='first')
    
    print(f"\n-> Final Breaks (Unmatched Status): {len(final_breaks_df)}")
    
    # Combine all matched, broken, and filtered transactions into the final output
    matched_and_broken_df = merged_df[
        (merged_df['Status'] != 'Unmatched') | (merged_df['Transaction ID'].isin(final_breaks_df['Transaction ID']))
    ].copy()
    
    # Add back the filtered transactions from Stage 1
    filtered_df = bank_df[bank_df['Enrichment Request status'] == 'Filtered - No Enrichment Required'].copy()
    filtered_df['Status'] = 'Filtered - No Match Attempted'
    
    final_output = pd.concat([
        matched_and_broken_df,
        filtered_df[['Transaction ID', 'Fund ID', 'Currency', 'Bank Amount', 'Status', 'Enrichment Request status']] 
    ], ignore_index=True)
    
    # Clean up and finalize columns for the Cash Control view
    output_cols = [
        'Transaction ID', 'Fund ID', 'Currency', 'Bank Amount', 'Book Amount',
        'Difference', 'Absolute Value Difference', 'Status', 'Enrichment Request status',
        'Break ID'
    ]
    
    # Final cleanup of NaN values for breaks/filtered items
    final_output = final_output[output_cols].fillna({
        'Book Amount': 0.00, 
        # For breaks, Difference = -Bank Amount (since Book is 0)
        'Difference': final_output.apply(lambda row: -row['Bank Amount'] if pd.isna(row['Book Amount']) else row['Difference'], axis=1),
        'Absolute Value Difference': final_output['Bank Amount']
    })
    
    # Correct Break ID for filtered and broken records
    final_output.loc[final_output['Break ID'].isnull(), 'Break ID'] = final_output.loc[
        final_output['Break ID'].isnull(), 'Break ID'
    ].apply(lambda x: str(uuid.uuid4()))
    
    return final_output


# --- 3. EXECUTION AND RESULTS DISPLAY ---

# Load Mock Data
bank_data_raw = create_mock_enriched_bank_data()
book_data_raw = create_mock_book_data()

# Run the Stage 2 Pipeline
cash_control_data = run_stage_2_matching(bank_data_raw, book_data_raw, FX_FEE_TOLERANCES)

print("\n--- STAGE 2 RESULTS: CASH CONTROL BREAKS & MATCHES ---")
# Show the records that were matched and the records that are breaks
print(cash_control_data.to_markdown(index=False))

# Summarize the final outcome
matched_count = len(cash_control_data[cash_control_data['Status'].str.contains('Matched')])
break_count = len(cash_control_data[cash_control_data['Status'] == 'Unmatched'])
filtered_count = len(cash_control_data[cash_control_data['Status'].str.contains('Filtered')])

print("\n--- RECONCILIATION SUMMARY ---")
print(f"Total Transactions Processed: {len(cash_control_data)}")
print(f"Auto-Matched Transactions: {matched_count}")
print(f"Filtered Transactions (Removed from Matching Scope): {filtered_count}")
print(f"Final Breaks (Manual Investigation Required): {break_count}")


-> Running 1. Exact Match Pass...
   - Found 2 exact matches.
-> Running 2. Fuzzy/Tolerance Match Pass (Bank Fees)...
   - Found 1 fee tolerance matches.
-> Running 3. Time-Lag Match Pass (3-day window)...
   - Found 2 time-lag matches.
-> Running 4. N:M Aggregation Match Pass (1:2 or 1:3)...
   - Searching for 1:N Aggregation Matches (1 Bank -> N Book)...
   - Found 1 aggregation matches.

-> Final Breaks (Unmatched Status): 3

--- STAGE 2 RESULTS: CASH CONTROL BREAKS & MATCHES ---
| Transaction ID   |   Fund ID | Currency   |   Bank Amount |   Book Amount |   Difference |   Absolute Value Difference | Status                         | Enrichment Request status   | Break ID                             |
|:-----------------|----------:|:-----------|--------------:|--------------:|-------------:|----------------------------:|:-------------------------------|:----------------------------|:-------------------------------------|
| BTXN01           |  50360414 | USD        |          9970 | 